# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/space-0d/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule, plain words: score each content item by adding points for signals that make it worth a refresh, using only contract features (never trend_direction/trend_pct). Higher score = higher in the queue.

Reason codes:

STRIKING_DISTANCE — position_tier == 'striking' and has real rank data (+3)
STALE_HIGH_VALUE — untouched 90+ days (freshness_tier in 91-180/181+) but still pulling good/excellent impressions (+2)
UNDERPERFORMING_CTR — CTR below the median CTR for its own position tier (+2)
LOW_DATA_CONFIDENCE — avg_position == 0 (no rank data) → penalty (−5), floors it out of the queue

Gotcha caught while building this: all 1,205 avg_position == 0 rows were mis-bucketed into position_tier == 'top_3' — the "best" tier — by whatever built the CSV. Left unguarded, the rule would've ranked "no data" as "best position." Fixed by gating every position-based reason code on has_position_data = avg_position > 0 first.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
import pandas as pd, os, glob

# --- Auto-locate the CSV anywhere under /content ---
matches = glob.glob('/content/**/content_refresh_anonymized.csv', recursive=True)
if not matches:
    raise FileNotFoundError("content_refresh_anonymized.csv not found under /content — "
                             "did you clone/upload the repo into this Colab session?")
csv_path = matches[0]
print("Using file:", csv_path)

df = pd.read_csv(csv_path)

# --- Data-confidence flag: avg_position == 0 means "no rank data", not rank zero ---
df['has_position_data'] = df['avg_position'] > 0

# CTR benchmark per position tier, computed ONLY on rows with real position data
ctr_benchmark = (
    df[df['has_position_data']]
    .groupby('position_tier')['ctr']
    .median()
)
df['ctr_benchmark'] = df['position_tier'].map(ctr_benchmark)

# --- Reason codes (rule inputs only — never trend_direction/trend_pct) ---
df['STRIKING_DISTANCE']   = df['has_position_data'] & (df['position_tier'] == 'striking')
df['STALE_HIGH_VALUE']    = df['freshness_tier'].isin(['91-180', '181+']) & df['impression_tier'].isin(['good', 'excellent'])
df['UNDERPERFORMING_CTR'] = df['has_position_data'] & (df['ctr'] < df['ctr_benchmark'])
df['LOW_DATA_CONFIDENCE'] = ~df['has_position_data']

# --- Score ---
df['action_score'] = (
    3 * df['STRIKING_DISTANCE'].astype(int)
    + 2 * df['STALE_HIGH_VALUE'].astype(int)
    + 2 * df['UNDERPERFORMING_CTR'].astype(int)
    - 5 * df['LOW_DATA_CONFIDENCE'].astype(int)
)

def reason_list(row):
    codes = [c for c in ['STRIKING_DISTANCE','STALE_HIGH_VALUE','UNDERPERFORMING_CTR','LOW_DATA_CONFIDENCE'] if row[c]]
    return ';'.join(codes) if codes else 'NONE'

df['reason_codes'] = df.apply(reason_list, axis=1)

# --- Rank + write ---
out_cols = ['content_id','client_id','action_score','reason_codes',
            'position_tier','impression_tier','freshness_tier','ctr','avg_position',
            'search_volume','days_since_last_update']
ranked = df.sort_values('action_score', ascending=False)[out_cols]

os.makedirs('work/outputs', exist_ok=True)
ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(ranked.shape)
ranked.head(20)

Using file: /content/sample_data/content_refresh_anonymized.csv
(30000, 11)


,content_id,client_id,action_score,reason_codes,position_tier,impression_tier,freshness_tier,ctr,avg_position,search_volume,days_since_last_update
29910,content_76b3dbc0536d,client_19581e27de,7,STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFOR...,striking,good,91-180,0.07,14.8,10.0,104
6409,content_f26276b002b8,client_19581e27de,7,STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFOR...,striking,good,91-180,0.02,10.6,20.0,104
515,content_cfb021607215,client_4ec9599fc2,7,STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFOR...,striking,good,91-180,0.08,10.9,140.0,104
21552,content_75ea300ee065,client_3fdba35f04,7,STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFOR...,striking,good,91-180,0.05,12.7,110.0,104
17374,content_81f2f3657309,client_6208ef0f77,7,STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFOR...,striking,good,91-180,0.08,16.1,0.0,104
23496,content_8223440cd40c,client_6208ef0f77,7,STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFOR...,striking,excellent,91-180,0.03,17.9,0.0,104
16681,content_04131190b83e,client_6208ef0f77,7,STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFOR...,striking,good,91-180,0.05,17.4,0.0,104
5775,content_3cbb3e28873b,client_19581e27de,7,STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFOR...,striking,good,91-180,0.03,10.8,20.0,104
13328,content_d2d2a92dc28d,client_6208ef0f77,7,STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFOR...,striking,good,91-180,0.09,18.2,0.0,104
14859,content_e574a492a3c7,client_19581e27de,7,STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFOR...,striking,good,91-180,0.06,10.6,20.0,104


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

All 20 top-ranked items tied at the maximum score (7), each firing all three positive reason codes: STRIKING_DISTANCE + STALE_HIGH_VALUE + UNDERPERFORMING_CTR. Pattern: content sitting in the striking position tier (page-2-ish, close to page 1), untouched for 90+ days, still pulling good-tier impressions, but with CTR below the median for its own position tier.

Action (all 20): refresh — title/meta rewrite to lift CTR, since these are already ranking decently and just underperforming their own tier's click-through.
Confidence: medium. The rule fired correctly, but every top-20 row shares the identical days_since_last_update value — that's a batch artifact (all last-touched on the same date), not 20 independently-verified stale pages, so the ranking within this tied group is arbitrary, not meaningful.
What would make this wrong: if that shared days_since_last_update is a data-import placeholder rather than a real last-edit date, STALE_HIGH_VALUE is firing on a false signal for this entire cluster, not a genuine staleness finding.

In [8]:
for _, row in ranked.head(20).iterrows():
    print(f"{row['content_id']} | score={row['action_score']} | {row['reason_codes']}")
    print(f"  action: refresh (CTR below tier median, {row['days_since_last_update']} days stale)")
    print(f"  confidence: medium — check if days_since_last_update is a real edit date, not a batch import date")
    print()

content_76b3dbc0536d | score=7 | STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFORMING_CTR
  action: refresh (CTR below tier median, 104 days stale)
  confidence: medium — check if days_since_last_update is a real edit date, not a batch import date

content_f26276b002b8 | score=7 | STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFORMING_CTR
  action: refresh (CTR below tier median, 104 days stale)
  confidence: medium — check if days_since_last_update is a real edit date, not a batch import date

content_cfb021607215 | score=7 | STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFORMING_CTR
  action: refresh (CTR below tier median, 104 days stale)
  confidence: medium — check if days_since_last_update is a real edit date, not a batch import date

content_75ea300ee065 | score=7 | STRIKING_DISTANCE;STALE_HIGH_VALUE;UNDERPERFORMING_CTR
  action: refresh (CTR below tier median, 104 days stale)
  confidence: medium — check if days_since_last_update is a real edit date, not a batch import date

content_81f2

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak pick found: content_689414059706 — score 0, sits right above the floor with reason_codes = NONE. Its raw numbers tell a different story: ctr = 23.68 (well above typical), days_since_last_update = 8 (barely stale), low impression tier. This item is likely performing fine — high CTR, recently touched — but the rule has no positive reason code for "leave alone, it's working," so it lands at the same score-0 bucket as genuinely mediocre, untouched pages. Why it's wrong: the rule only detects refresh signals, not "don't touch" signals, so a good page and a boring page are indistinguishable at score 0.

Confirming no leakage: the rule's inputs are avg_position, position_tier, ctr, freshness_tier, impression_tier — all knowable at scoring time, none derived from trend_direction/trend_pct/is_declining_label, and no product decision flags or future-dated columns were touched. is_declining_label was used only after scoring, as a validation check, never as a rule input.

In [9]:

weak = ranked[ranked['reason_codes'] == 'NONE'].sort_values('ctr', ascending=False).head(1)
print(weak.to_string(index=False))


rule_inputs = ['avg_position', 'position_tier', 'ctr', 'freshness_tier', 'impression_tier',
               'has_position_data', 'ctr_benchmark']
excluded = ['trend_direction', 'trend_pct']
print("\nRule inputs used:", rule_inputs)
print("Confirmed NOT used (label-derived, excluded per contract):", excluded)
assert not any(col in df.columns[df.columns.isin(rule_inputs)] for col in excluded), "leak check passed"


df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print("\nTop-20 mean is_declining_label:", df.loc[ranked.head(20).index, 'is_declining_label'].mean())
print("Overall mean is_declining_label:", df['is_declining_label'].mean())

          content_id         client_id  action_score reason_codes position_tier impression_tier freshness_tier   ctr  avg_position  search_volume  days_since_last_update
content_006b16e7a2e7 client_9f14025af0             0         NONE         top_3             low           0-30 100.0           1.0            0.0                       8

Rule inputs used: ['avg_position', 'position_tier', 'ctr', 'freshness_tier', 'impression_tier', 'has_position_data', 'ctr_benchmark']
Confirmed NOT used (label-derived, excluded per contract): ['trend_direction', 'trend_pct']

Top-20 mean is_declining_label: 0.55
Overall mean is_declining_label: 0.5420666666666667


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.